# StructuralPro — آزمون مستقل و رایگان PR #481

**نسخه مخصوص Google Colab (بدون GitHub Actions، کامپیوتر ویندوزی، کارت بانکی یا توکن GitHub)**

این دفترچه فقط از مخزن عمومی `Asanda021/StructuralPro` و شاخه `feat/finish-assemblies-ui-boq-20261011` کد می‌خواند و در یک محیط موقت اجرا می‌کند. هر بار آخرین SHA همان PR از GitHub گرفته و با Clone تطبیق داده می‌شود.

**اجرا از گوشی:** در Colab از منوی **Runtime → Run all** استفاده کن. اتصال به اینترنت و محیط CPU گوگل لازم است. در صورت عدم دسترسی به Colab، از منوی File → Download استفاده کن و برای بررسی در اختیار من بگذار.

**ممنوع:** ساخت EXE، Installer، APK، Release، Merge و Push. این دفترچه هیچ‌یک از این کارها را انجام نمی‌دهد. فقط آزمون Python/Qt (حالت offscreen)، ذخیره و گزارش‌های مصنوعی را اجرا می‌کند. هیچ اطلاعات واقعی پروژه یا Secret وارد نکن.

**اعتبار نتیجه:** اگر حتی یک آزمون اجرا نشده یا خطا داده باشد، «تأیید کامل» محسوب نمی‌شود. گزارش SHA، لاگ‌های اجرا و فایل‌های مصنوعی برای بررسی ارائه می‌شوند.


In [ ]:
# ۱) دریافت شاخه واقعی PR از مخزن عمومی با تطبیق دقیق SHA
from pathlib import Path
import os, sys, subprocess, json, urllib.request, shutil, platform, time

REPO = 'Asanda021/StructuralPro'
BRANCH = 'feat/finish-assemblies-ui-boq-20261011'
PR = 481
WORK = Path('/content/structuralpro_pr481_checkout')
OUT = Path('/content/structuralpro_pr481_qa_results')
OUT.mkdir(parents=True, exist_ok=True)

def command(args, *, cwd=None, env=None, log_name=None):
    print('\n▶', ' '.join(map(str, args)), flush=True)
    p = subprocess.run(args, cwd=cwd, env=env, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True)
    msg = p.stdout or '(بدون خروجی)'
    print(msg[-20000:], flush=True)
    if log_name:
        (OUT/log_name).write_text(msg, encoding='utf-8')
    print('Exit code:', p.returncode, flush=True)
    return p.returncode

req = urllib.request.Request(
    f'https://api.github.com/repos/{REPO}/pulls/{PR}',
    headers={'Accept':'application/vnd.github+json', 'User-Agent':'structuralpro-qa-colab'})
with urllib.request.urlopen(req, timeout=35) as response:
    pr = json.load(response)
EXPECTED = pr['head']['sha']
assert pr['head']['ref'] == BRANCH, 'شاخه PR تغییر کرده است؛ برای ایمنی متوقف شد.'
assert pr['state'] == 'open', 'PR دیگر Open نیست؛ اجرای خودکار متوقف شد.'
print('Expected PR HEAD:', EXPECTED)
if WORK.exists():
    shutil.rmtree(WORK)
clone_code = command(['git', 'clone', '--quiet', '--depth', '1', '--single-branch',
                      '--branch', BRANCH, f'https://github.com/{REPO}.git', str(WORK)],
                      log_name='git_clone.txt')
assert clone_code == 0, 'دریافت مخزن ناموفق است؛ تست اجرا نشده.'
ACTUAL = subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK,text=True).strip()
assert ACTUAL == EXPECTED, f'HEAD تغییر کرده: {ACTUAL} != {EXPECTED}; تست اجرا نشد.'
print('✅ PR SHA matched:', ACTUAL)
(OUT/'source_revision.json').write_text(json.dumps({
    'repository':REPO, 'pr':PR, 'branch':BRANCH,
    'sha':ACTUAL, 'python':sys.version, 'platform':platform.platform(),
    'started_at_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'synthetic_only':True,
},indent=2,ensure_ascii=False),encoding='utf-8')
os.environ['QT_QPA_PLATFORM']='offscreen'
os.environ['PYTHONPATH']=str(WORK)
print('CPU-only. No build or release actions are performed.')


In [ ]:
# ۲) نصب وابستگی‌های آزمون فقط در ماشین موقت Colab
assert WORK.is_dir(), 'مرحله دریافت مخزن باید اول موفق شود.'
packages = command(['apt-get','update','-qq'], log_name='apt_update.log')
if packages == 0:
    packages = command(['apt-get','install','-y','-qq','libegl1','libopengl0','libxkbcommon-x11-0'],
                       log_name='apt_qt_libs.log')
pip_result = command([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],
                     cwd=WORK,log_name='pip_install.log')
assert packages == 0 and pip_result == 0, 'نصب وابستگی کامل نشد؛ آزمون سبز اعلام نشود.'
q = command([sys.executable,'-c',
 'import pytest, fitz, docx, openpyxl, reportlab, arabic_reshaper; from bidi.algorithm import get_display; from PySide6.QtWidgets import QApplication; app=QApplication([]); print("Qt offscreen ready")'],
 cwd=WORK,env=os.environ.copy(),log_name='import_runtime.log')
assert q == 0, 'محیط Qt یا گزارش‌ها قابل بارگیری نیست.'


In [ ]:
# ۳) اجرای آزمون‌های واقعی، بدون توقف زودهنگام بین مجموعه‌ها
assert WORK.is_dir() and (WORK/'tests/test_finish_takeoff_assemblies.py').is_file()
TARGET_SUITES = {
  'finishing_core_storage_boq': ['tests/test_finish_takeoff_assemblies.py'],
  'qt_offscreen': ['tests/test_finish_takeoff_ui_runtime.py'],
  'report_xlsx_docx_pdf': ['tests/test_finish_takeoff_report_contract.py'],
  'legacy_manual_regression': [
    'tests/test_master13_phase2_manual_transaction.py',
    'tests/test_master13_phase2_manual_dialog_runtime.py',
    'tests/test_manual_takeoff_workbench.py',
    'tests/test_manual_takeoff_input.py',
    'tests/test_manual_takeoff_ui_wiring.py',
  ],
}
results = {}
for suite, paths in TARGET_SUITES.items():
    missing = [p for p in paths if not (WORK/p).is_file()]
    if missing:
        print('⚠️ فایل آزمون موجود نیست:',suite,missing)
        results[suite] = {'returncode':None, 'missing':missing, 'passed':False}
        continue
    log = f'{suite}.log'
    rc = command([sys.executable,'-m','pytest','-q','-ra','--disable-warnings', *paths],
                 cwd=WORK,env=os.environ.copy(),log_name=log)
    results[suite] = {'returncode':rc,'missing':[], 'passed':rc == 0, 'log':log}
(OUT/'test_results.json').write_text(json.dumps(results,indent=2,ensure_ascii=False),encoding='utf-8')
print('\nخلاصه:',json.dumps(results,indent=2,ensure_ascii=False))
if not all(x['passed'] for x in results.values()):
    print('⚠️ بخشی از آزمون‌ها FAILED/MISSING هستند. نتیجه را سبز اعلام نکن.')
else:
    print('✅ همه مجموعه‌های انتخاب‌شده با exit-code صفر تمام شدند؛ Skipped را در لاگ‌ها بررسی کن.')


In [ ]:
# ۴) تولید شواهد مصنوعی فارسی فقط در صورت موفقیت همه مجموعه‌های آزمون
assert 'results' in globals(), 'ابتدا آزمون‌ها را اجرا کن.'
qa_output = OUT/'synthetic_evidence'
if all(x['passed'] for x in results.values()):
    qa_output.mkdir(parents=True, exist_ok=True)
    rc = command([sys.executable,'tools/qa/finish_report_preview.py','--output',str(qa_output)],
                 cwd=WORK,env=os.environ.copy(),log_name='synthetic_evidence.log')
    if rc != 0:
        print('⚠️ تولید شواهد گزارش/تصویر شکست خورد. آزمون نهایی مورد تأیید نیست.')
else:
    (OUT/'synthetic_evidence_not_executed.txt').write_text(
      'Core/Qt/report regressions were not all successful; synthetic report evidence was deliberately skipped.',
      encoding='utf-8')
    print('⏭️ شواهد تصویری اجرا نشدند تا خروجی ناقص به‌جای گزارش معتبر ارائه نشود.')


In [ ]:
# ۵) بسته نتیجه: فقط لاگ و شواهد مصنوعی؛ برای ارسال به ChatGPT
import zipfile, hashlib, json
assert OUT.is_dir() and (OUT/'source_revision.json').is_file(), 'ابتدا مرحله ۱ را اجرا کن.'
artifact = Path('/content/StructuralPro_PR481_Colab_QA_Results.zip')
with zipfile.ZipFile(artifact,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUT.rglob('*')):
        if file.is_file():
            z.write(file,file.relative_to(OUT))
print('نتایج قابل دریافت:',artifact, 'bytes:',artifact.stat().st_size)
print('SHA-256:',hashlib.sha256(artifact.read_bytes()).hexdigest())
try:
    from google.colab import files
    files.download(str(artifact))
except ImportError:
    print('این دفترچه خارج از Colab اجرا شده؛ ZIP از مسیر بالا قابل دریافت است.')
